[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Sets


## What you will be able to do

Use a **set** to remove duplicates in one step, ask whether something is present without
waiting for a search, and compare two collections to see what they share and what only one of
them has.


## The idea

### The problem

Suppose you have five thousand survey responses and want to know how many **different** people
answered.

With a list, you would walk through the responses, keep a second list of everyone seen so far,
and for each name check whether it is already in that second list. That check is itself a
search through everything collected to date, so the work grows as the list grows. It is a lot
of code, and it gets slow, for a question that is easy to state.

Now suppose you have yesterday's attendees and today's, and you want the people who came on
both days. That is a loop inside a loop. Then you want the people who came yesterday but not
today. Another one.

### What a set is

> A **set** is a collection with two rules: **no duplicates** and **no order**. Adding a value
> that is already present does nothing at all. Items must be **unchangeable**, the same
> restriction that applies to dictionary keys.
>
> Curly braces create one, and `set()` converts anything you can loop over.

### What those two rules buy

**Duplicates cannot exist.** Not "are removed for you", but cannot exist. Building a set from a
list of five thousand responses gives you the distinct ones as a side effect of construction,
in one call, with no loop.

**Membership stays fast.** This is the part worth understanding. To find something in a list,
Python compares against each item in turn, so a longer list means a longer wait. A set instead
computes, from the value itself, where that value would have to be stored, and looks in exactly
that one place. The work does not grow with the size of the set. Ten items or ten million, the
same time.

That is not a small difference and this notebook measures it rather than asserting it.

**Comparisons become single operations.** In both lists, in one but not the other, in either:
each is one symbol, replacing a loop.

### What it costs

A set throws away two things, and they are not recoverable afterward.

It forgets **how many times** it saw each value, so it cannot answer "which response was most
common". It forgets **order**, so there is no first item, no position, and no promise that
printing it twice gives the same arrangement.

When you need counts, notebook 9 is the right tool. When you need order, notebook 7 is.

### Where you will meet this

Removing duplicates from imported data. Testing membership repeatedly inside a loop, where the
speed matters. Comparing two collections: which required fields are missing from a form, which
records appear in both files, which values are new since the last run.

### What this notebook covers

- Creating sets, and why `{}` is not an empty one
- Removing duplicates, and converting back to a list in a sensible order
- That there is genuinely no order, demonstrated in a way you can reproduce
- Adding and removing, and the two methods that differ only on a miss
- Membership speed, measured across three sizes
- Union, intersection, difference and symmetric difference
- Subset tests, which are the tidy way to check required fields
- When a set is the wrong choice
- Three errors, and what each one is telling you

Start with duplicates collapsing.


In [1]:
numbers = {3, 1, 4, 1, 5, 9, 2, 6, 5, 3}

print(numbers)
print(len(numbers), "unique values from ten")


{1, 2, 3, 4, 5, 6, 9}
7 unique values from ten


Nothing was chosen or discarded by you. A set simply cannot hold the same value twice, so
adding a duplicate does nothing at all.

That single rule is what makes the rest of this notebook useful.


## Setup

One import, used only for the timing near the end.


In [2]:
import time

print("Ready.")


Ready.


## Worked examples

### Making one, and the empty-set trap

Curly braces make a set, except when they make a dictionary.


In [3]:
print(type({1, 2, 3}))
print(type({}))          # not an empty set
print(type(set()))       # this is the empty set


<class 'set'>
<class 'dict'>
<class 'set'>


`{}` was a dictionary before sets existed, and it stayed one. The empty set has to be written
`set()`.

`set()` also converts anything you can loop over.


In [4]:
print(set([1, 2, 2, 3]))
print(set("mississippi"))


{1, 2, 3}
{'i', 'm', 'p', 's'}


### Removing duplicates, which is why most people reach for a set

One call, and the order is gone with them.


In [5]:
visits = ["ada", "grace", "ada", "alan", "grace", "ada"]

unique = set(visits)

print(unique)
print(len(visits), "visits from", len(unique), "people")


{'grace', 'ada', 'alan'}
6 visits from 3 people


When you need a list back, convert it. When you need it in a sensible order, `sorted` gives a
list directly.


In [6]:
print(list(unique))
print(sorted(unique))


['grace', 'ada', 'alan']
['ada', 'alan', 'grace']


### There is genuinely no order

This is not "the order is different from what you put in". It is "there is no order, and you
must not rely on what you see".


In [7]:
print(set("zebra"))


{'z', 'e', 'r', 'a', 'b'}


**Your output above is probably not the same as the one saved in this notebook, and running the
cell again in a fresh session may give a third answer.** That is not a bug and it is worth
seeing: Python deliberately scrambles how text is stored between runs.

Small whole numbers are the exception that misleads people. They tend to come out looking
sorted, which is a coincidence of how numbers are stored and not a promise.


In [8]:
print(set([3, 1, 2, 10, 7]))     # looks sorted, is not guaranteed to be


{1, 2, 3, 7, 10}


If order matters, you want a list. If you want a set **and** an order, sort it when you take it
out.

### Adding and removing


In [9]:
letters = {"a", "b", "c"}

letters.add("d")
print(sorted(letters))

letters.discard("z")     # not there, and that is fine
letters.remove("a")      # not there would raise, see Common errors
print(sorted(letters))


['a', 'b', 'c', 'd']
['b', 'c', 'd']


`discard` and `remove` do the same thing, and differ only when the item is absent: `discard`
shrugs, `remove` raises. Choose the one that matches whether a miss is expected, which is the
same decision as `get` against `[]` in notebook 9.

Note that `add` and `remove` change the set in place and return `None`, exactly like the list
methods in notebook 7.


### Membership, and why it stays fast

`in` works as it does everywhere else.


In [10]:
people = {"ada", "grace", "alan"}

print("ada" in people)
print("bob" in people)


True
False


The difference is what happens underneath. To find something in a **list**, Python looks at the
items one after another, so a longer list means a longer wait. A **set** computes where the
item would be and looks there, which takes the same time no matter how much is in it.

Here is that difference measured, asking for the worst case each time: an item at the very end.


In [11]:
print(f"{'items':>10}  {'list':>12}  {'set':>10}")

for n in (1_000, 10_000, 100_000):
    as_list = list(range(n))
    as_set = set(as_list)
    target = n - 1                      # the last item, worst case for a list
    reps = 2000 if n <= 10_000 else 300

    start = time.perf_counter()
    for _ in range(reps):
        target in as_list
    list_us = (time.perf_counter() - start) / reps * 1e6

    start = time.perf_counter()
    for _ in range(reps):
        target in as_set
    set_us = (time.perf_counter() - start) / reps * 1e6

    print(f"{n:>10,}  {list_us:>10.1f}us  {set_us:>8.2f}us")


     items          list         set
     1,000         4.6us      0.03us
    10,000        37.1us      0.03us


   100,000       341.6us      0.03us

The exact numbers depend on the machine and will differ from the ones saved here. The shape
will not: the list column grows roughly in step with the number of items, and the set column
stays flat.

This is the reason to convert a list to a set before checking membership repeatedly. For a
single check it does not matter; inside a loop over thousands of items it is the difference
between instant and unusable.


### Comparing two sets

This is where sets stop being a container and start being a tool. Four operations, each with a
symbol and a method name.

| Symbol | Method | Gives |
|---|---|---|
| `a \| b` | `a.union(b)` | in either |
| `a & b` | `a.intersection(b)` | in both |
| `a - b` | `a.difference(b)` | in `a` only |
| `a ^ b` | `a.symmetric_difference(b)` | in one but not both |


In [12]:
monday = {"ada", "grace", "alan"}
tuesday = {"grace", "alan", "bob"}

print("either day     ", sorted(monday | tuesday))
print("both days      ", sorted(monday & tuesday))
print("monday only    ", sorted(monday - tuesday))
print("exactly one day", sorted(monday ^ tuesday))


either day      ['ada', 'alan', 'bob', 'grace']
both days       ['alan', 'grace']
monday only     ['ada']
exactly one day ['ada', 'bob']


Every one of those would be a loop with an `if` inside it otherwise. Written this way the code
says what it means.

The methods do the same thing and accept any collection, not only a set, which is sometimes
handier.


In [13]:
print(sorted(monday.intersection(["grace", "bob"])))


['grace']


### Asking about the whole set

`<=` asks whether everything on the left is also on the right.


In [14]:
required = {"name", "email"}
provided = {"name", "email", "phone"}

print(required <= provided)              # is everything required present
print(required.issubset(provided))       # the same question
print(provided - required)               # what extra was provided
print(required.isdisjoint({"phone"}))    # nothing in common


True
True
{'phone'}
True


Checking that a form has every required field is a real use of `<=`, and it beats a loop with a
counter.

### What can go in a set

The same rule as dictionary keys in notebook 8: items must be unchangeable. Text, numbers and
tuples are fine. Lists are not.


In [15]:
points = {(0, 0), (3, 7), (0, 0)}

print(points)
print(len(points), "unique points from three")


{(3, 7), (0, 0)}
2 unique points from three


### When a set is the wrong choice

A set throws away two things, and sometimes you needed them.

| You lose | So do not use a set when |
|---|---|
| Duplicates | You need to count how many times something appeared |
| Order | The sequence matters, or you need positions |

For counting, a dictionary from notebook 9 is the right tool.


In [16]:
visits = ["ada", "grace", "ada", "alan", "grace", "ada"]

print("set says how many people: ", len(set(visits)))

counts = {}
for person in visits:
    counts[person] = counts.get(person, 0) + 1
print("dictionary says how often:", counts)


set says how many people:  3
dictionary says how often: {'ada': 3, 'grace': 2, 'alan': 1}


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/10-sets-solutions.ipynb).

Use these for tasks 1 to 4:

```
morning = ["tea", "toast", "eggs", "tea", "juice"]
evening = ["soup", "toast", "tea", "bread"]
```

**1.** Print how many different items appear in `morning`.


In [17]:
# your code here


**2.** Print the items that appear in both lists, sorted.


In [18]:
# your code here


**3.** Print the items that appear in `morning` but not in `evening`, sorted.


In [19]:
# your code here


**4.** Print every item across both lists, sorted, with no repeats.


In [20]:
# your code here


**5.** Given `required = {"name", "email", "age"}` and `form = {"name", "email"}`, print
whether the form has everything required, then print what is missing.


In [21]:
# your code here


**6.** Given `"the quick brown fox jumps over the lazy dog"`, print how many different letters
it uses, ignoring spaces.


In [22]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### KeyError: remove needs something that is there


In [23]:
letters = {"a", "b"}
letters.remove("z")


KeyError: 'z'

`KeyError: 'z'` is the same complaint a dictionary makes for a missing key, because a set is
built the same way underneath.

Use `discard` when the item may not be there:


In [24]:
letters = {"a", "b"}
letters.discard("z")     # no error

print(sorted(letters))


['a', 'b']


### TypeError: a set has no positions

There is no order, so there is nothing for a position to mean.


In [25]:
letters = {"a", "b", "c"}
print(letters[0])


TypeError: 'set' object is not subscriptable

`'set' object is not subscriptable` is Python saying the question does not apply. If you want
the first of something, you want an ordered container: convert with `sorted()` and take from
that.


In [26]:
print(sorted(letters)[0])


a


### TypeError: a list cannot go in a set

The rule from notebook 8, in its other form.


In [27]:
{[1, 2], [3, 4]}


TypeError: cannot use 'list' as a set element (unhashable type: 'list')

`unhashable type: 'list'` means the item could change, and a set has to know where each item
lives. Tuples are the fix, exactly as they were for dictionary keys.


In [28]:
print({(1, 2), (3, 4)})


{(1, 2), (3, 4)}


## Recap

- A set holds unique items with no order, so `set(items)` removes duplicates in one step.
- `{}` is an empty **dictionary**. The empty set is `set()`.
- Printing order is not insertion order and can change between runs; sort it if you care.
- `in` on a set takes the same time however large the set is, unlike a list.
- `|` `&` `-` `^` give union, intersection, difference and symmetric difference.
- `<=` asks whether one set is contained in another, which is the tidy way to check required
  fields.
- Set items must be unchangeable, so tuples work and lists do not.
- A set discards counts and order. When you need those, use a dictionary or a list.


## What is next

**Notebook 11, Conditionals**, where the true and false answers from notebook 6 start deciding
which code runs at all: `if`, `elif`, `else`, and how to keep them from nesting into a mess.


---

&#8592; **Previous:** [Dictionaries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/09-dictionaries.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)  &nbsp;·&nbsp;  **Next:** [Conditionals](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/11-conditionals.ipynb) &#8594;
